# **Deep Learning Methods in Image Processing and Computer Vision**

### 1. Feature generation with Convolution network

### 2. Capsule Network based Image classification

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Capsule Layer
class CapsuleLayer(layers.Layer):
    def __init__(self, num_capsules, dim_capsules, num_routing):
        super(CapsuleLayer, self).__init__()
        self.num_capsules = num_capsules
        self.dim_capsules = dim_capsules
        self.num_routing = num_routing

    def build(self, input_shape):
        self.W = self.add_weight(shape=[self.num_capsules, input_shape[1], self.dim_capsules, input_shape[2]],
                                 initializer='glorot_uniform', trainable=True)

    def call(self, inputs):
        inputs_expand = tf.expand_dims(inputs, 1)
        inputs_tiled = tf.tile(inputs_expand, [1, self.num_capsules, 1, 1])
        inputs_hat = tf.map_fn(lambda x: tf.matmul(self.W, x), elems=inputs_tiled)
        b = tf.zeros(shape=[tf.shape(inputs_hat)[0], self.num_capsules, tf.shape(inputs_hat)[2]])
        for i in range(self.num_routing):
            c = tf.nn.softmax(b, axis=1)
            outputs = tf.reduce_sum(c * inputs_hat, axis=2)
            if i != self.num_routing - 1:
                b += tf.reduce_sum(inputs_hat * tf.expand_dims(outputs, -1), axis=-1)
        return tf.sqrt(tf.reduce_sum(tf.square(outputs), axis=-1) + tf.keras.backend.epsilon())

# Build CapsNet Model
def build_capsnet(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    conv1 = layers.Conv2D(256, (9, 9), activation='relu')(inputs)
    primary_caps = layers.Conv2D(256, (9, 9), strides=2, activation='relu')(conv1)
    primary_caps = layers.Reshape((-1, 8))(primary_caps)
    digit_caps = CapsuleLayer(num_capsules=num_classes, dim_capsules=16, num_routing=3)(primary_caps)
    outputs = layers.Lambda(lambda z: tf.sqrt(tf.reduce_sum(tf.square(z), axis=-1)))(digit_caps)
    return models.Model(inputs, outputs)

# Instantiate and compile the model
model = build_capsnet(input_shape=(28, 28, 1), num_classes=10)
model.compile(optimizer='adam', loss='margin_loss', metrics=['accuracy'])

### 3. DCGAN for image generation

### 4. CycleGAN Image to Image Translation

### 5. SRGAN Super Resolution

### 6. Conditional GAN and Info GAN

### 7. YOLO V5 / Yolo V6 / Mask R-CNN / Center Mask Object Detection

### 8. Panoptic / Instance / Semantic Segmentation

### 9. Face Arithmetic with GAN vs VAE

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Conv2D, Conv2DTranspose, LeakyReLU
from tensorflow.keras.models import Model
import numpy as np

# Define the generator model
def build_generator(latent_dim):
    model = tf.keras.Sequential()
    model.add(Dense(128 * 8 * 8, activation="relu", input_dim=latent_dim))
    model.add(Reshape((8, 8, 128)))
    model.add(Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2D(3, kernel_size=7, activation="tanh", padding="same"))
    return model

# Define the discriminator model
def build_discriminator(img_shape):
    model = tf.keras.Sequential()
    model.add(Conv2D(64, kernel_size=4, strides=2, input_shape=img_shape, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Conv2D(128, kernel_size=4, strides=2, padding="same"))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Flatten())
    model.add(Dense(1, activation='sigmoid'))
    return model

# Define the GAN model
def build_gan(generator, discriminator):
    discriminator.trainable = False
    gan_input = Input(shape=(latent_dim,))
    img = generator(gan_input)
    gan_output = discriminator(img)
    gan = Model(gan_input, gan_output)
    gan.compile(loss='binary_crossentropy', optimizer='adam')
    return gan

latent_dim = 100
img_shape = (64, 64, 3)

generator = build_generator(latent_dim)
discriminator = build_discriminator(img_shape)
discriminator.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

gan = build_gan(generator, discriminator)

# Example of face arithmetic
def face_arithmetic(latent_vector1, latent_vector2, operation='add'):
    if operation == 'add':
        result_vector = latent_vector1 + latent_vector2
    elif operation == 'subtract':
        result_vector = latent_vector1 - latent_vector2
    generated_face = generator.predict(np.expand_dims(result_vector, axis=0))
    return generated_face

# Example usage
latent_vector1 = np.random.normal(size=(latent_dim,))
latent_vector2 = np.random.normal(size=(latent_dim,))
generated_face = face_arithmetic(latent_vector1, latent_vector2, operation='add')

In [ ]:
from tensorflow.keras.layers import Input, Dense, Lambda, Conv2D, Flatten, Conv2DTranspose, Reshape
from tensorflow.keras.models import Model
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras import backend as K
import numpy as np

# Define the encoder model
def build_encoder(img_shape, latent_dim):
    inputs = Input(shape=img_shape)
    x = Conv2D(32, kernel_size=3, strides=2, padding='same', activation='relu')(inputs)
    x = Conv2D(64, kernel_size=3, strides=2, padding='same', activation='relu')(x)
    x = Flatten()(x)
    x = Dense(128, activation='relu')(x)
    z_mean = Dense(latent_dim)(x)
    z_log_var = Dense(latent_dim)(x)
    z = Lambda(sampling, output_shape=(latent_dim,))([z_mean, z_log_var])
    return Model(inputs, [z_mean, z_log_var, z])

# Define the decoder model
def build_decoder(latent_dim, img_shape):
    latent_inputs = Input(shape=(latent_dim,))
    x = Dense(8 * 8 * 64, activation='relu')(latent_inputs)
    x = Reshape((8, 8, 64))(x)
    x = Conv2DTranspose(64, kernel_size=3, strides=2, padding='same', activation='relu')(x)
    x = Conv2DTranspose(32, kernel_size=3, strides=2, padding='same', activation='relu')(x)
    outputs = Conv2DTranspose(3, kernel_size=3, padding='same', activation='sigmoid')(x)
    return Model(latent_inputs, outputs)

# Sampling function
def sampling(args):
    z_mean, z_log_var = args
    batch = K.shape(z_mean)[0]
    dim = K.int_shape(z_mean)[1]
    epsilon = K.random_normal(shape=(batch, dim))
    return z_mean + K.exp(0.5 * z_log_var) * epsilon

# Define the VAE model
def build_vae(encoder, decoder, img_shape):
    inputs = Input(shape=img_shape)
    z_mean, z_log_var, z = encoder(inputs)
    outputs = decoder(z)
    vae = Model(inputs, outputs)
    reconstruction_loss = binary_crossentropy(K.flatten(inputs), K.flatten(outputs)) * np.prod(img_shape)
    kl_loss = -0.5 * K.sum(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var), axis=-1)
    vae_loss = K.mean(reconstruction_loss + kl_loss)
    vae.add_loss(vae_loss)
    vae.compile(optimizer='adam')
    return vae

latent_dim = 100
img_shape = (64, 64, 3)

encoder = build_encoder(img_shape, latent_dim)
decoder = build_decoder(latent_dim, img_shape)
vae = build_vae(encoder, decoder, img_shape)

# Example of face arithmetic
def face_arithmetic(latent_vector1, latent_vector2, operation='add'):
    if operation == 'add':
        result_vector = latent_vector1 + latent_vector2
    elif operation == 'subtract':
        result_vector = latent_vector1 - latent_vector2
    generated_face = decoder.predict(np.expand_dims(result_vector, axis=0))
    return generated_face

# Example usage
latent_vector1 = np.random.normal(size=(latent_dim,))
latent_vector2 = np.random.normal(size=(latent_dim,))
generated_face = face_arithmetic(latent_vector1, latent_vector2, operation='add')

### Comparison: VAE vs GAN

### 10. Fast-AgeGAN

### 11. Conditional VAE

### 12. Text to Image Synthesis with GAN

### 13. DeepFake / DeepFaceLab

### 13. Show attend and tell – Natural Language Caption generation for images

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# CNN Encoder
def build_encoder():
    base_model = applications.InceptionV3(include_top=False, weights='imagenet')
    base_model.trainable = False
    return models.Model(base_model.input, base_model.layers[-1].output)

# Attention Mechanism
class BahdanauAttention(tf.keras.layers.Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = layers.Dense(units)
        self.W2 = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, features, hidden):
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))
        attention_weights = tf.nn.softmax(self.V(score), axis=1)
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector, attention_weights

# RNN Decoder
class RNNDecoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size):
        super(RNNDecoder, self).__init__()
        self.units = units
        self.embedding = layers.Embedding(vocab_size, embedding_dim)
        self.gru = layers.GRU(self.units, return_sequences=True, return_state=True, recurrent_initializer='glorot_uniform')
        self.fc1 = layers.Dense(self.units)
        self.fc2 = layers.Dense(vocab_size)
        self.attention = BahdanauAttention(self.units)

    def call(self, x, features, hidden):
        context_vector, attention_weights = self.attention(features, hidden)
        x = self.embedding(x)
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)
        output, state = self.gru(x)
        x = self.fc1(output)
        x = tf.reshape(x, (-1, x.shape[2]))
        x = self.fc2(x)
        return x, state, attention_weights

# Instantiate and compile the model
encoder = build_encoder()
decoder = RNNDecoder(embedding_dim=256, units=512, vocab_size=5000)

### 15. Deep reinforcement learning for image processing

### Chapter: DALL-E – Text to Image Generation

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import GPT2Tokenizer, GPT2Model

# Text Encoder
class TextEncoder(nn.Module):
    def __init__(self, model_name='gpt2'):
        super(TextEncoder, self).__init__()
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.model = GPT2Model.from_pretrained(model_name)

    def forward(self, text):
        inputs = self.tokenizer(text, return_tensors='pt')
        outputs = self.model(**inputs)
        return outputs.last_hidden_state

# Image Decoder
class ImageDecoder(nn.Module):
    def __init__(self, embed_dim, image_size):
        super(ImageDecoder, self).__init__()
        self.fc = nn.Linear(embed_dim, image_size * image_size * 3)
        self.image_size = image_size

    def forward(self, embeddings):
        x = self.fc(embeddings)
        x = x.view(-1, 3, self.image_size, self.image_size)
        return x

# DALL-E Model
class DALL_E(nn.Module):
    def __init__(self, text_encoder, image_decoder):
        super(DALL_E, self).__init__()
        self.text_encoder = text_encoder
        self.image_decoder = image_decoder

    def forward(self, text):
        text_embeddings = self.text_encoder(text)
        image = self.image_decoder(text_embeddings[:, 0, :])
        return image

# Instantiate and train the model
text_encoder = TextEncoder()
image_decoder = ImageDecoder(embed_dim=768, image_size=64)
model = DALL_E(text_encoder, image_decoder)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Dummy training loop
for epoch in range(10):
    text = ["A cat sitting on a mat"]
    target_image = torch.randn(1, 3, 64, 64)  # Dummy target image

    optimizer.zero_grad()
    output_image = model(text)
    loss = criterion(output_image, target_image)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")